# Customer Churn Analysis and Prediction Using Data Analytics and Machine Learning

## 1. Project Title
**Customer Churn Analysis and Prediction Using Data Analytics and Machine Learning**

## 2. Project Overview
This project analyzes customer data to identify patterns associated with churn and builds a machine learning model to predict whether a customer is likely to leave.

## 3. Problem Statement
Customer churn is a critical metric for businesses. Identifying factors that lead to customer attrition allows companies to proactively address issues and improve retention rates.

## 4. Objectives
- Perform Exploratory Data Analysis (EDA) on customer data.
- Identify key features influencing churn.
- Build and evaluate a Machine Learning model (Logistic Regression) to predict churn.

## 5. Dataset Description
The dataset used is the Telco Customer Churn dataset, containing demographics, services, account information, and churn status.

## 6. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

## 7. Load Dataset

In [ ]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## 8. Data Understanding

In [ ]:
print(f"Shape of dataset: {df.shape}")
df.info()

In [ ]:
df.describe()

## 9. Data Cleaning

In [ ]:
# Handle empty spaces in TotalCharges
df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

# Drop rows with missing TotalCharges
df = df.dropna(subset=['TotalCharges'])
print(f"Dataset shape after cleaning: {df.shape}")

# Drop customerID
df = df.drop('customerID', axis=1)

## 10. Exploratory Data Analysis & 11. Data Visualization

In [ ]:
sns.set_theme(style="whitegrid")

# Churn distribution
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Churn', palette='Set2')
plt.title('Churn Distribution')
plt.show()

In [ ]:
# Churn by Contract Type
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='Contract', hue='Churn', palette='Set2')
plt.title('Churn by Contract Type')
plt.show()

In [ ]:
# Monthly Charges Distribution by Churn
plt.figure(figsize=(8,5))
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', multiple="stack", palette='Set2', bins=30)
plt.title('Monthly Charges Distribution by Churn')
plt.show()

## 12. Feature Engineering & 13. Data Preprocessing

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
    ]
)

## 14. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

## 15. Machine Learning Model

In [ ]:
lr_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

lr_model.fit(X_train, y_train)

## 16. Model Evaluation

In [ ]:
y_pred = lr_model.predict(X_test)
y_pred_prob = lr_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, y_pred_prob):.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()

## 17. Feature Importance / Interpretation

In [ ]:
feature_names = num_cols + list(lr_model.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_cols))
coefficients = lr_model.named_steps['classifier'].coef_[0]

feat_imp = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})
feat_imp['Abs_Coefficient'] = feat_imp['Coefficient'].abs()
feat_imp = feat_imp.sort_values(by='Abs_Coefficient', ascending=False).head(10)

plt.figure(figsize=(10,6))
sns.barplot(data=feat_imp, x='Coefficient', y='Feature', palette='coolwarm')
plt.title('Top 10 Feature Coefficients (Logistic Regression)')
plt.show()

## 18. Example Churn Prediction

In [ ]:
example_data = pd.DataFrame([X_test.iloc[0]])
example_pred = lr_model.predict(example_data)
example_prob = lr_model.predict_proba(example_data)[:, 1]

print(f"Input features:\n{example_data.to_dict(orient='records')[0]}")
print(f"Predicted Churn: {'Yes' if example_pred[0] == 1 else 'No'}")
print(f"Churn Probability: {example_prob[0]:.4f}")

## 19. Key Insights
- Customers with Month-to-month contracts are significantly more likely to churn compared to one-year or two-year contracts.
- Higher monthly charges are generally associated with a higher likelihood of churn.
- Certain payment methods, specifically electronic check, are associated with a higher churn rate.
- Tenure is negatively correlated with churn (longer tenure = less likely to churn).

## 20. Conclusion
The Logistic Regression model successfully predicts customer churn with reasonable accuracy. The analysis reveals that contract type, tenure, and monthly charges are strong predictors of churn. Businesses can use these insights to target high-risk customers with retention offers, such as incentives to switch to longer-term contracts.

## 21. Future Enhancements
- Hyperparameter tuning using GridSearchCV.
- Experimenting with tree-based models like Random Forest and XGBoost.
- Deploying the model as a REST API using Flask or FastAPI.
- Building an interactive dashboard for real-time churn monitoring.